# Adım 4: Keşifsel Veri Analizi (EDA)
**Kişi 2 sorumluluğu** — `feature/spark-eda` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, avg
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

GOLD_PATH = './delta_lake/gold'
PLOTS_DIR = './plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateEDA')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .config('spark.sql.execution.arrow.pyspark.enabled', 'false')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
df = spark.read.format('delta').load(GOLD_PATH)
print(f'Gold tablodan {df.count():,} kayit yuklendi.')

In [ ]:
pdf = df.select('avg_temp_c', 'min_temp_c', 'max_temp_c',
                'precipitation_mm', 'avg_wind_speed_kmh').describe().toPandas()
print('[EDA] Temel Istatistikler:')
print(pdf.to_string())

In [ ]:
cols = ['avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm',
        'snow_depth_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa',
        'sunshine_total_min']
missing = {c: df.filter(col(c).isNull() | isnan(col(c))).count() for c in cols}
total = df.count()
missing_pct = {k: round(v / total * 100, 2) for k, v in missing.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(missing_pct.keys(), missing_pct.values(),
              color=sns.color_palette('Reds_r', len(cols)))
ax.set_title('Sutun Bazli Eksik Deger Orani (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Eksik Deger (%)')
ax.set_xlabel('Sutunlar')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/01_missing_values.png')
plt.show()
print('[Gorsel 1] Eksik deger grafigi kaydedildi.')